# 07 — Train model open-set (tiered)

Nhánh **đánh giá / open-set**: rừng + tầng L1 + ngưỡng theo `n_sources`, contract `sdc-tiered-v2`. Sau bước này chạy `07_evaluate_openset` → `08_test_openset` → `10_activate_openset` → `11_export_onnx_openset`.

Nhánh **production** (`05_train_closedset` → `09_build_sdc_iden`) không phụ thuộc bước này.

In [ ]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd


## Tạo candidate open-set (tiered)

Đầu vào là `Data/sessions_verified.parquet` từ `03_build_dataset` / `04_add_capture_devices`. Chỉ bật `REBUILD_VERIFIED` nếu cần dựng lại bản đã xác minh từ `Data/sessions.parquet` và taxonomy. Train ghi artifact vào `Models/<run_id>/`; chưa kích hoạt model phục vụ — đó là việc của `10_activate_openset`.

In [26]:
RUN_ID = None
DATE_FOLDS = 5
SKIP_DEVICE_OOF = False
REUSE_OOF_FROM = None
REBUILD_VERIFIED = False


In [27]:
from IPython.display import display

if REBUILD_VERIFIED:
    build_verified_dataset(
        source=ROOT / "Data" / "sessions.parquet",
        labels_path=ROOT / "Data" / "device_labels_verified.csv",
        excluded_path=ROOT / "Data" / "device_labels_excluded.csv",
        output=SESSIONS_PATH,
    )
sessions, feature_groups = load_sessions(SESSIONS_PATH)
print(f"Dataset: {len(sessions)} session, {sessions.canonical_device.nunique()} thiết bị")

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn\.utils\.parallel")

train_args = ["--sessions", str(SESSIONS_PATH), "--date-folds", str(DATE_FOLDS)]
if RUN_ID:
    train_args += ["--run-id", RUN_ID]
if SKIP_DEVICE_OOF:
    train_args += ["--skip-device-oof"]
if REUSE_OOF_FROM:
    train_args += ["--reuse-oof-from", str(Path(REUSE_OOF_FROM))]
run_dir = train_main(train_args).resolve()
predictor = Predictor(run_dir, enrolled={})
print("Candidate:", run_dir)
print("Contract:", predictor.contract_format)
display(pd.read_csv(run_dir / "thresholds.csv"))


Dataset: 8189 session, 51 thiết bị
training 20260915_132703_verified_tiered: 8189 sessions, 51 devices, 40 features
id_date fold 1/5
id_date fold 2/5
id_date fold 3/5
id_date fold 4/5
id_date fold 5/5
unseen_class make: 1/26 - Amazon
unseen_class make: 2/26 - Amcrest
unseen_class make: 3/26 - Apple
unseen_class make: 4/26 - Arlo
unseen_class make: 5/26 - Borun
unseen_class make: 6/26 - Camera
unseen_class make: 7/26 - D-Link
unseen_class make: 8/26 - Eufy
unseen_class make: 9/26 - Generic Laptop
unseen_class make: 10/26 - Google
unseen_class make: 11/26 - HeimVision
unseen_class make: 12/26 - Home Eye
unseen_class make: 13/26 - LG
unseen_class make: 14/26 - Linova/Linux
unseen_class make: 15/26 - Luohe
unseen_class make: 16/26 - Netatmo
unseen_class make: 17/26 - OPPO
unseen_class make: 18/26 - Philips
unseen_class make: 19/26 - Raspberry Pi
unseen_class make: 20/26 - Ring
unseen_class make: 21/26 - Samsung
unseen_class make: 22/26 - SimCam
unseen_class make: 23/26 - Sonos
unseen_class

,threshold,id_coverage,id_accuracy_answered,ood_abstain,n_id,n_ood,target_accuracy,target_reached,objective,overridden,head,n_sources
0,1.00,0.987642,1.0,0.925561,2023,2230,0.99,True,0.913203,False,make,1
1,0.96,0.985370,1.0,0.999772,2529,4392,0.99,True,0.985142,False,make,2
2,0.92,0.963883,1.0,0.995839,886,1442,0.99,True,0.959722,False,make,3
3,0.48,0.968750,1.0,0.976000,96,125,0.99,True,0.944750,False,make,4
4,1.00,0.985707,1.0,1.000000,2029,2229,0.90,True,0.985707,False,type,1
5,1.00,0.960652,1.0,1.000000,2516,4392,0.90,True,0.960652,False,type,2
6,0.72,0.986456,1.0,0.947989,886,1442,0.90,True,0.934445,False,type,3
7,0.80,0.791667,1.0,1.000000,96,125,0.90,True,0.791667,True,type,4
8,1.00,0.985171,1.0,0.925561,2023,2230,0.99,True,0.910731,False,model,1
9,1.00,0.959272,1.0,1.000000,2529,4392,0.99,True,0.959272,False,model,2
